# Retrieving NBA Player Biometric Data Using the nba_api Library

This notebook documents the exact process we followed to retrieve biometric attributes of NBA players for every season (2003-04 through 2023-24). 

The data was pulled using the official nba_api Python wrapper for NBA.coms endpoints. It was designed to produce one row per player per season containing only the fields we needed for joining with shot data later in the pipeline. After retrieval, the data was consolidated and published to Kaggle so our main Colab notebook can load it directly without repeated API calls.

Final output structure per season (matches what we wanted):
- PLAYER_ID, PLAYER_NAME, AGE, PLAYER_HEIGHT_INCHES, PLAYER_WEIGHT, DRAFT_YEAR, SEASON_1

## Data Retrieval Plan and Notes (from our project documentation)

- We needed player-specific biometric data per season to join with the Kaggle shot dataset[](https://www.kaggle.com/datasets/mexwell/nba-shots).
- No pre-existing dataset matched our exact requirements (player ID/name + season + biometrics only), so we built our own using the nba_api library.
- Endpoint used: LeagueDashPlayerBioStats (returns bio stats for every player who appeared in a given season).
- Pulled locally once and saved as CSVs to avoid rate limits, future API changes, and repeated calls in Colab.
- After cleaning (dropping advanced stats, team info, etc.), we added a SEASON_1 column for easy joining.
- The resulting per-season CSVs were combined into a single dataset and uploaded to our public Kaggle repository.

Kaggle Dataset (for easy access in Colab):
Biometric Attributes of NBA Players per Season[](https://www.kaggle.com/datasets/joefodera/biometric-attributes-of-nba-players-per-season/data)

This Kaggle link is the canonical source we now import in our main analysis Colab.

## Things Learned During Implementation

- nba_api is a stable, actively maintained wrapper (repo was updated shortly before our pull).
- Rate limiting is real. TIME_BETWEEN_CALLS = 30 seconds prevents blocks.
- The endpoint returns exactly one DataFrame per season call.
- We kept only the biometric fields required; dropped everything else (advanced metrics, college, country, draft round/number, etc.).
- Draft year is included (as requested in planning). Undrafted players appear naturally (DRAFT_YEAR is blank/NaN).
- Wingspan and years-of-experience were intentionally omitted (not universally available or required extra API complexity).
- Final pipeline note: keep SEASON_1 for stratification but do not train on it.

## Exact Python Script Used (nba-api-pull.py)

The block below is 100 percent identical to the script we ran locally. No changes were made.

To re-run in Colab (optional): Add this in a cell above it:
pip install nba_api pandas

The script will create 21 CSV files (NBA_2004_Players.csv  NBA_2024_Players.csv).

In [ ]:
# What we get from API: 
##['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT']
# What we want as output (no specific order):
##[PLAYER_ID,PLAYER_NAME,AGE,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,DRAFT_YEAR,SEASON_1]
### Structure of SEASON_1 = '2024' 

# User Notes
## You may need to play with TIME_BETWEEN_CALLS to prevent rate limit blocking from the NBA 

import time
import pandas as pd
from nba_api.stats.endpoints import LeagueDashPlayerBioStats

# Display Settings
pd.set_option('display.max_columns', None)

# Var init 
SEASONS_TO_PULL = [
    "2003-04", "2004-05", "2005-06", "2006-07", "2007-08",
    "2008-09", "2009-10", "2010-11", "2011-12", "2012-13",
    "2013-14", "2014-15", "2015-16", "2016-17", "2017-18",
    "2018-19", "2019-20", "2020-21", "2021-22", "2022-23",
    "2023-24"
]
COLS_TO_DROP = ['TEAM_ID', 'TEAM_ABBREVIATION', 'PLAYER_HEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT']
TIME_BETWEEN_CALLS = 30

seasonIndex = 0
for season1 in range(2004, 2025):
  # Asking "return every player with the season=season1"
  allFromSeason = LeagueDashPlayerBioStats(
    season=SEASONS_TO_PULL[seasonIndex]
  )

  currentSeasonFrame = allFromSeason.get_data_frames()[0]   # 0 because this specific endpoint only returns one results set 
  currentSeasonFrame = currentSeasonFrame.drop(columns = COLS_TO_DROP)
  currentSeasonFrame['SEASON_1'] = season1
  currentSeasonFrame.to_csv('NBA_' + str(season1) + '_Players.csv', index=False)
  print('NBA_' + str(season1) + '_Players.csv has been successfully retrieved and created. \nIt pulled from the ' + SEASONS_TO_PULL[seasonIndex] + ' Season')
  seasonIndex += 1
  time.sleep(TIME_BETWEEN_CALLS)

print('Data retrieval complete!!')  


## Results and Next Steps

Script successfully generated one CSV per season (2004-2024 labeling, pulled from the corresponding 2003-04 to 2023-24 NBA seasons).

All files contain exactly the columns we needed.

Data was merged into a single dataset and published to the Kaggle link above.

You can now import the Kaggle dataset directly in any Colab notebook using the standard Kaggle API or pd.read_csv after downloading.

This completes the biometric data retrieval portion of the project. The rest of the pipeline (joining with shot data, dropping identifiers, VIF checks, modeling) continues from the consolidated Kaggle file.

Reference: Full planning notes are in data-pulling.md (included in the project repo).